In [12]:
import fs from 'node:fs';
import path from 'node:path';
import { ChatOpenAI } from '@langchain/openai';
import { OPENAI_API_KEY } from './src/lib/vars.mjs';
import * as z from 'zod';

function logFile(logContent, fileName = 'jupyter.md') {
  const logFileName = `logs/${fileName}`;
  const logDir = path.dirname(logFileName);
  if (!fs.existsSync(logDir)) {
    fs.mkdirSync(logDir, { recursive: true });
  }
  fs.writeFileSync(logFileName, '```markdown\n' + logContent + '\n```', 'utf8');
  return logContent;
}

const gpt5 = new ChatOpenAI({
  modelName: 'gpt-5',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt4oMini = new ChatOpenAI({
  modelName: 'gpt-4o-mini',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});
const gpt5Nano = new ChatOpenAI({
  modelName: 'gpt-5-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt41Nano = new ChatOpenAI({
  modelName: 'gpt-4.1-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});

async function queryAllModels(prompt, callFunction) {
  console.log(`Started: ${new Date()}\n`);
  return Promise.all([
    callFunction(prompt, gpt4oMini),
    callFunction(prompt, gpt41Nano),
    callFunction(prompt, gpt5Nano),
    callFunction(prompt, gpt5),
  ]);
}

## Rewrite Initial Query


### Sanitize Query

**Results:** 

All models performed satisfactory with similar times.

In [ ]:
import { rewriteQuery } from './src/lib/rag.mjs';

const redditQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`;

async function sanitizeQuery(question, model) {
  const { answer: query } = await rewriteQuery(question, model);
  console.log(`**${model.model}** – ${new Date()}:\n${query}\n`);
  return query;
}

await queryAllModels(redditQuestion, sanitizeQuery);


Started: Fri Aug 22 2025 13:03:57 GMT-0400 (Eastern Daylight Time)



Promise { <pending> }

### Query Extraction

#### Decomposing Main Query

#### Test Results

| Model          | Response Time | Performance | Notes                                                   |
| -------------- | ------------- | ----------- | ------------------------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        |                                                         |
| GPT-4.1-nano   | < 5 sec       | Good        |                                                         |
| GPT-5-nano     | < 30 sec      | Mediocre    | Questions need further break down.                      |
| GPT-5          | > 1 min       | Ok          | Questions well-thought but must be broken down further. |


In [3]:
const compoundQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`;

const questionExtractionPrompt = `You are an assistant that prepares user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration rules.
The user may provide a long, informal story or question. Your task is:
1. Identify all explicit and implicit questions they are asking.  
2. Rewrite each one as a clear, self-contained question that could be answered directly from IRCC documentation.  
3. Condense the result into the *smallest possible set of non-overlapping, atomic questions* that fully capture the user’s intent.  
4. Eliminate redundancy — avoid rephrasing the same issue multiple times.  
5. Do not provide answers — only the minimal list of questions.

**User question:**

\`\`\`
${compoundQuestion}
\`\`\`
`;

async function extractQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z
          .array(z.string())
          .describe('A list of questions derived from the user query, sorted by relevance top to bottom.'),
      })
    )
    .invoke(q);
  const content = response.questions.map((q) => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

const extractedQuestions = await queryAllModels(questionExtractionPrompt, extractQuestions);


Started: Fri Aug 22 2025 13:03:57 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** Fri Aug 22 2025 13:03:58 GMT-0400 (Eastern Daylight Time):
	- Can an exchange student enter Canada on a tourist visa and switch to a study permit after arrival?
	- What are the requirements and procedures for changing from a tourist visa to a study permit in Canada?
	- Will entering Canada on a tourist visa affect future eligibility for a study permit or other visas?
	- Is it possible to travel back to Canada after visiting with a tourist visa if the initial entry was on a tourist visa?
	- What are the risks and implications of entering Canada with a tourist visa when planning to study there?

**gpt-4o-mini** Fri Aug 22 2025 13:03:59 GMT-0400 (Eastern Daylight Time):
	- What are the processing times for a study permit application for Canada?
	- Can I enter Canada on a tourist visa while waiting for my study permit?
	- Will entering Canada on a tourist visa affect my ability to return after leaving for 

#### Identifying Key Questions

##### Test Summary

| Model           | Response Time | Performance | Notes                                  |
| --------------- | ------------- | ----------- | -------------------------------------- |
| 🥇 GPT-4.1-nano | < 5 sec       | Good        | Correctly identified the key question. |
| GPT-4o-mini     | < 5 sec       | Good        | Included some secondary questions.     |
| GPT-5-nano      | < 30 sec      | Mediocre    | Included the most questions.           |
| GPT-5           | < 30 sec      | Ok          | Included some secondary questions.     |


In [4]:

const mdExtractedQuestions = extractedQuestions[1].questions.map(q => `\t- ${q}`).join('\n');
const questionDiscriminationPrompt = `You are helping prepare user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration.
Input: a list of atomic questions generated from a user’s long query.
Task:
1. Identify the key question(s) that directly capture the user’s main intent.  
   - Keep only the questions that must be answered to resolve the user’s core concern.  
   - Discard questions that are secondary, conditional, or only relevant as follow-ups.
2. Output only the minimal set of key questions, without explanation, ranked by relevance to the user query.
Important: The result should be as short as possible while still fully representing the original user’s primary intent.

User query:

\`\`\`
${compoundQuestion}
\`\`\`

List of questions:
\`\`\`
${mdExtractedQuestions}
\`\`\`
`

async function discriminateQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z.array(z.string()).describe('A list of key questions derived from the user query'),
      })
    )
    .invoke(q);
  const content = response.questions.map(q => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

console.log(`\n\nDiscriminating questions for: "${compoundQuestion}"\n`);
console.log(`Extracted questions:\n${mdExtractedQuestions}\n`);
await queryAllModels(questionDiscriminationPrompt, discriminateQuestions);



Discriminating questions for: "Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if th

[
  {
    questions: [
      "Can an exchange student enter Canada on a tourist visa and switch to a study permit after arrival?",
      "Will entering Canada on a tourist visa affect future eligibility for a study permit or other visas?",
      "What are the risks and implications of entering Canada with a tourist visa when planning to study there?"
    ]
  },
  {
    questions: [
      "Can an exchange student enter Canada on a tourist visa and switch to a study permit after arrival?"
    ]
  },
  {
    questions: [
      "Can an exchange student enter Canada on a tourist visa and switch to a study permit after arrival?",
      "What are the requirements and procedures for changing from a tourist visa to a study permit in Canada?",
      "Will entering Canada on a tourist visa affect future eligibility for a study permit or other visas?"
    ]
  }
]

## Vector Search

### Retrieval

In [5]:
import { vectorSearch, chunksToMarkdown } from './src/lib/vector-search.mjs';
const retrieveQuery = 'Can I enter Canada on a tourist visa while waiting for my study permit?'
const chunks = await vectorSearch(retrieveQuery);
logFile(chunksToMarkdown(chunks), 'chunks.md');
chunks

[
  {
    text: "# Guide 5269 - Applying for a Study Permit outside Canada\n" +
      "\n" +
      "[Print](javascript:window.print\\(\\);)\n" +
      "\n" +
      "## You need a provincial attestation letter (PAL) or territorial attestation letter (TAL) to apply for a study permit\n" +
      "\n" +
      "Most students must include with their study permit application a PAL/TAL from the province or territory where they plan to study.\n" +
      "\n" +
      "In most cases, if you apply without a PAL/TAL, your application will be returned with fees.\n" +
      "\n" +
      "[Learn more about the provincial attestation letter and territorial attestation letter](/en/immigration-refugees-citizenship/services/study-canada/study-permit/get-documents/provincial-attestation-letter.html).\n" +
      "\n" +
      "**Francophone Minority Communities Student Pilot (FMCSP)**\n" +
      "\n" +
      "To be eligible you must:\n" +
      "\n" +
      "\\[...\\]\n" +
      "\n" +
      "* * * \n" +
   

### Reference Discrimination

#### Test Summary

| Model          | Response Time | Performance | Notes                                 |
| -------------- | ------------- | ----------- | ------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        | Picked the most chunks, all relevant. |
| GPT-4.1        | < 5 sec       | Ok          |                                       |
| GPT-5          | < 1 min       | Ok          |                                       |
| GPT-5-nano     | < 30 sec      | Mediocre    | Missed a highly relevant chunk.       |


In [6]:
const chunkDiscriminationPrompt = `I'll give you a markdown content with a list of results from a vector search for a question.
Select only the references that help to answer th question.
Pay attention to the header of the top-most header in each reference to identify if it's related to the topic we want to answer; discard it if it is not.
Return an array containing the selected references' numbers.

**Question:** ${retrieveQuery}

**Chunks:**

\`\`\`markdown
${chunksToMarkdown(chunks)}
\`\`\`
`;

async function discriminateReferences(prompt, model) {
  const { references: referenceIndexes } = await model
  .withStructuredOutput(
    z.object({
      references: z.array(z.number()).describe('An array of numbers representing the selected references from the chunks'),
    })
  )
  .invoke(prompt);

  console.log(`**${model.model}** – ${new Date()}: ${JSON.stringify(referenceIndexes)}`);
  return referenceIndexes;
}

const referenceIndexes = await queryAllModels(chunkDiscriminationPrompt, discriminateReferences);

Started: Fri Aug 22 2025 13:04:31 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Fri Aug 22 2025 13:04:32 GMT-0400 (Eastern Daylight Time): [2,5]
**gpt-4o-mini** – Fri Aug 22 2025 13:04:32 GMT-0400 (Eastern Daylight Time): [2,4,5,6]
**gpt-5-nano** – Fri Aug 22 2025 13:04:43 GMT-0400 (Eastern Daylight Time): [2,4]


## Answering

### Generating Answer

#### Test Summary

| Model          | Response Time | Performance | Notes                             |
| -------------- | ------------- | ----------- | --------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        | Concise and accurate.             |
| GPT-5-nano     | < 30 sec      | Good        | Good structure. Too many details. |
| GPT-4.1        | < 5 sec       | Mediocre    | Rarely provides sources.          |
| GPT-5          | < 1 min       | Ok          | Good structure. Too many details. |


In [7]:
import { generateAnswer } from './src/lib/rag.mjs';

async function generateRAGAnswer(query, model) {
  const selectChunks = chunks.filter((_, i) => referenceIndexes[1].includes(i + 1));
  const references = chunksToMarkdown(selectChunks);
  const response = await generateAnswer(query, references, model);
  const { answer } = response;
  console.log(
    `**${model.model}** – ${new Date()}:\n${answer}\n\n* * *\n`
  );
  return response;
}

console.log(`"${retrieveQuery}"\n`);
const generatedResponses = await queryAllModels(retrieveQuery, generateRAGAnswer);

"Can I enter Canada on a tourist visa while waiting for my study permit?"

Started: Fri Aug 22 2025 13:04:43 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Fri Aug 22 2025 13:04:45 GMT-0400 (Eastern Daylight Time):
Yes, it is possible to enter Canada on a visitor visa (or with an Electronic Travel Authorization, eTA) while waiting for your study permit, but final entry decisions are made by the border officer at the port of entry. If you already have a valid visitor visa or eTA, you can use it to travel to Canada. However, if your study permit application is still in process, you cannot study until your new permit is approved and issued. Additionally, if your study permit has expired, you may need to re-apply or restore your status before re-entering. It's important to carry all necessary documents, including proof of your application and your travel documents, and be prepared for the border officer to assess your situation at the point of entry. Potential nuances include whether

## Evaluating Answer

### Test Summary
...

In [15]:
const answer = generatedResponses[0].answer;
const evaluationPrompt = `Evaluate if the answer provided by a RAG bot is fully addressing the user's concerns.

Input: 
1. The user's original question.
2. The answer generated by the RAG bot.

Your task:
- Determine if the answer fully addresses the user's question.
- If the answer is incomplete, does not address the user's concerns, or is irrelevant, return an array with a single required follow up questions to ask the RAG bot.
- The questions must be ordered by relevance, with the most important question first.
- If the answer is complete and directly addresses the user's concerns, return an empty array.

**User's question:**

\`\`\`
${redditQuestion}
\`\`\`

**RAG bot's answer:**

\`\`\`
${answer}
\`\`\`
`;

async function discriminateReferences(prompt, model) {
  const { questions } = await model
  .withStructuredOutput(
    z.object({
      questions: z.array(z.string()).describe('An array containing 1 question as input to the RAG bot'),
    })
  )
  .invoke(prompt);

  console.log(`**${model.model}** – ${new Date()}: ${JSON.stringify(questions, null, 2)}\n\n`);
  return questions;
}

console.log(`—"${redditQuestion}"\n\n—"${answer}"\n\n* * *\n\n`);
const followUpQuestions = await queryAllModels(evaluationPrompt, discriminateReferences);

—"Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to re